# Model Definitions


In [ ]:
import os
import sys

for site in [
    os.path.join(os.environ.get("LOCALAPPDATA", ""), "Programs", "Python", "Python314", "Lib", "site-packages"),
    os.path.join(os.environ.get("APPDATA", ""), "Python", "Python314", "site-packages")
]:
    if os.path.exists(site) and site not in sys.path:
        sys.path.insert(0, site)

os.environ.setdefault("KERAS_HOME", ".")

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

try:
    from xgboost import XGBClassifier
except ImportError:
    XGBClassifier = None

IMG_SIZE = 64
CHANNELS = 3

def build_ann():
    model = keras.Sequential([
        layers.Input(shape=(IMG_SIZE, IMG_SIZE, CHANNELS)),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(1, activation="sigmoid")
    ], name="ann")
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

def build_cnn():
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, CHANNELS))

    # block 1
    c1 = layers.Conv2D(32, 3, padding='same', activation='relu')(inputs)
    c1 = layers.BatchNormalization()(c1)
    c1 = layers.Conv2D(32, 3, padding='same', activation='relu')(c1)
    c1 = layers.BatchNormalization()(c1)
    s1 = layers.Conv2D(32, 1, padding='same')(inputs)
    x = layers.MaxPooling2D(2)(layers.add([c1, s1]))

    # block 2
    c2 = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    c2 = layers.BatchNormalization()(c2)
    c2 = layers.Conv2D(64, 3, padding='same', activation='relu')(c2)
    c2 = layers.BatchNormalization()(c2)
    s2 = layers.Conv2D(64, 1, padding='same')(x)
    x = layers.MaxPooling2D(2)(layers.add([c2, s2]))

    # block 3
    c3 = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    c3 = layers.BatchNormalization()(c3)
    c3 = layers.Conv2D(128, 3, padding='same', activation='relu')(c3)
    c3 = layers.BatchNormalization()(c3)
    s3 = layers.Conv2D(128, 1, padding='same')(x)
    x = layers.MaxPooling2D(2)(layers.add([c3, s3]))

    # block 4 + head
    x = layers.Conv2D(256, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(64, activation="relu")(x)
    out = layers.Dense(1, activation="sigmoid")(x)

    model = keras.Model(inputs=inputs, outputs=out, name="residual_cnn")
    model.compile(optimizer=keras.optimizers.Adam(3e-4), loss="binary_crossentropy", metrics=["accuracy"])
    return model

def build_rnn():
    model = keras.Sequential([
        layers.Input(shape=(IMG_SIZE, IMG_SIZE, CHANNELS)),
        layers.Reshape((IMG_SIZE, IMG_SIZE * CHANNELS)),
        layers.Bidirectional(layers.LSTM(64)),
        layers.Dropout(0.3),
        layers.Dense(32, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ], name="rnn")
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

def build_transformer():
    patch_size = 8
    num_patches = (IMG_SIZE // patch_size) ** 2
    dim = 64

    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, CHANNELS))
    patches = layers.Conv2D(dim, patch_size, strides=patch_size, padding="valid")(inputs)
    x = layers.Reshape((num_patches, dim))(patches)
    
    norm1 = layers.LayerNormalization(epsilon=1e-6)(x)
    attn = layers.MultiHeadAttention(num_heads=4, key_dim=dim)(norm1, norm1)
    x = layers.add([attn, x])
    
    norm2 = layers.LayerNormalization(epsilon=1e-6)(x)
    mlp = layers.Dense(dim * 2, activation="relu")(norm2)
    mlp = layers.Dropout(0.2)(mlp)
    mlp = layers.Dense(dim)(mlp)
    x = layers.add([mlp, x])

    x = layers.GlobalAveragePooling1D()(x)
    out = layers.Dense(1, activation="sigmoid")(x)

    model = keras.Model(inputs=inputs, outputs=out, name="vit")
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

def build_xgboost():
    if XGBClassifier is None:
        raise ImportError("xgboost not installed")
    return XGBClassifier(
        n_estimators=120,
        max_depth=5,
        learning_rate=0.08,
        reg_lambda=1.0,
        n_jobs=-1,
        random_state=42
    )

